# HTML / DOM Phishing Classifier

Fine-tunes **DistilBERT** on crawled page HTML to classify URLs as **phishing (1)** or **benign (0)**.

### Pipeline overview
```
OpenPhish feed  ──→ crawl HTML ──┐
                                 ├──→ DOM extraction ──→ CSV ──→ S3
malicious_phish.csv (benign) ───┘         │
                                           ▼
                         Amazon SageMaker AI HuggingFace Estimator
                         (distilbert-base-uncased, ml.g5.2xlarge)
                                           │
                                           ▼
                         SageMaker AI HuggingFace Endpoint
                         (ml.m5.xlarge, text-classification)
```

## 0 · Install dependencies

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas scikit-learn boto3 sagemaker

## 1 · Configuration

In [ ]:
import hashlib
import json
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import os
import boto3
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode, InputData
from sagemaker.core.image_uris import retrieve as retrieve_image_uri
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

sess   = Session()
REGION = boto3.session.Session().region_name
ROLE   = get_execution_role()


BUCKET = sess.default_bucket()          # or replace with your bucket name
PREFIX = 'html-classifier'
CACHE_DIR = os.path.join(os.getcwd(), 'html_source')  # Cache for HTML data capture
DATA_DIR = os.path.join(os.getcwd(), 'data')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


PHISHING_TARGET = 5_000    # target page count per class
BENIGN_TARGET   = 5_000
MAX_WORKERS      = 50      # parallel threads
CRAWL_TIMEOUT    = 8       # seconds per URL

print(f'DATA_DIR: {DATA_DIR}')
print(f'CACHE_DIR: {CACHE_DIR}')
print(f'Region : {REGION}')
print(f'Role   : {ROLE}')
print(f'Bucket : {BUCKET}')

## 2 · Crawl HTML

### 2.1 · Crawl helpers

In [ ]:
from utils import crawl_benign_pages

### 2.2 · Crawl phishing pages

In [ ]:
PHISHING_SCENARIOS = [
    {'brand': 'PayPal',           'attack': 'account suspended',         'domain': 'paypal.com'},
    {'brand': 'Chase Bank',       'attack': 'suspicious login attempt',  'domain': 'chase.com'},
    {'brand': 'Wells Fargo',      'attack': 'account verification',      'domain': 'wellsfargo.com'},
    {'brand': 'Bank of America',  'attack': 'unusual activity detected', 'domain': 'bankofamerica.com'},
    {'brand': 'Citibank',         'attack': 'card blocked',              'domain': 'citibank.com'},
    {'brand': 'Apple',            'attack': 'Apple ID locked',           'domain': 'apple.com'},
    {'brand': 'Microsoft',        'attack': 'account security alert',    'domain': 'microsoft.com'},
    {'brand': 'Google',           'attack': 'sign-in blocked',           'domain': 'google.com'},
    {'brand': 'Amazon',           'attack': 'order payment issue',       'domain': 'amazon.com'},
    {'brand': 'Dropbox',          'attack': 'shared file access',        'domain': 'dropbox.com'},
    {'brand': 'Facebook',         'attack': 'account disabled',          'domain': 'facebook.com'},
    {'brand': 'Instagram',        'attack': 'copyright violation',       'domain': 'instagram.com'},
    {'brand': 'LinkedIn',         'attack': 'profile action required',   'domain': 'linkedin.com'},
    {'brand': 'Netflix',          'attack': 'payment failed',            'domain': 'netflix.com'},
    {'brand': 'Spotify',          'attack': 'subscription expired',      'domain': 'spotify.com'},
    {'brand': 'DHL',              'attack': 'package delivery failed',   'domain': 'dhl.com'},
    {'brand': 'FedEx',            'attack': 'parcel held at customs',    'domain': 'fedex.com'},
    {'brand': 'IRS',              'attack': 'tax refund available',      'domain': 'irs.gov'},
    {'brand': 'Coinbase',         'attack': 'wallet verification',       'domain': 'coinbase.com'},
    {'brand': 'eBay',             'attack': 'seller account warning',    'domain': 'ebay.com'},
]

# Variation hints keep generated pages diverse so the model doesn't overfit to one template
VARIATION_HINTS = [
    'Make the page very urgent with a 24-hour deadline.',
    'Include a fake case/ticket number and customer support email.',
    'Add fake security trust badges and SSL indicators.',
    'Include a fake countdown timer in the body text.',
    'Make it look like an automated email converted to a webpage.',
]

SAMPLES_PER_SCENARIO = 50      # 20 scenarios × 50 = 1000 synthetic pages
BEDROCK_MODEL_ID     = 'us.anthropic.claude-sonnet-4-6'
BEDROCK_MAX_WORKERS  = 20      # parallel Amazon Bedrock calls — stay within service TPS

bedrock_rt = boto3.client('bedrock-runtime', region_name=REGION)


def generate_phishing_html(scenario: dict, variation: int = 0) -> str | None:
    """Call Amazon Bedrock to generate one synthetic phishing HTML page."""
    brand  = scenario['brand']
    attack = scenario['attack']
    domain = scenario['domain']
    hint   = VARIATION_HINTS[variation % len(VARIATION_HINTS)]
    slug   = brand.lower().replace(' ', '-')
    fake_action = f'http://secure-{slug}-verify.phishtest.invalid/submit'

    prompt = (
        f'You are generating synthetic HTML for training a phishing DETECTION classifier '
        f'(defensive security research — classifier training only).\n\n'
        f'Generate a realistic phishing HTML page impersonating {brand}.\n'
        f'Attack scenario: {attack}\n'
        f'Legitimate domain being spoofed: {domain}\n\n'
        f'Requirements:\n'
        f'1. <title> must mention {brand}\n'
        f'2. A <form method="POST" action="{fake_action}"> — posts to a fake external URL\n'
        f'3. Input fields for email/username AND password\n'
        f'4. <meta name="description"> content mentioning {brand}\n'
        f'5. At least 2 <script> or <img> tags loading from different fake external domains\n'
        f'6. Convincing visible body text for the "{attack}" scenario\n'
        f'7. Style note: {hint}\n\n'
        f'All URLs MUST end in .invalid or .example (non-routable) — they must not be real.\n'
        f'Return only raw HTML. No markdown, no code fences, no explanation, absolutely NO emojis, the dataset will be used to train a model so please dont make obvious samples such as phish.invalid at the end of a sample url'
    )

    try:
        resp = bedrock_rt.converse(
            modelId=BEDROCK_MODEL_ID,
            messages=[{'role': 'user', 'content': [{'text': prompt}]}],
            inferenceConfig={'maxTokens': 1500, 'temperature': 0.9},
        )
        return resp['output']['message']['content'][0]['text']
    except Exception as e:
        print(f'  [gen] {brand} v{variation}: {e}')
        return None


def generate_all_phishing(
    scenarios: list,
    samples_per_scenario: int,
    cache_dir: str = CACHE_DIR,
    max_workers: int = BEDROCK_MAX_WORKERS,
) -> list:
    """
    Generate synthetic phishing pages for all scenarios.
    Caches each page to disk — re-runs skip already-generated pages.
    """
    def _gen_one(scenario, variation):
        key        = f'synthetic_{scenario["brand"].lower().replace(" ", "_")}_{variation}'
        cache_path = os.path.join(cache_dir, f'{key}.json')
        if os.path.exists(cache_path):
            with open(cache_path) as fh:
                return json.load(fh)
        html   = generate_phishing_html(scenario, variation)
        record = {
            'url':   f'https://{scenario["brand"].lower().replace(" ", "-")}-phish.invalid',
            'html':  html,
            'label': 1,
        }
        with open(cache_path, 'w') as fh:
            json.dump(record, fh)
        return record

    tasks = [(s, v) for s in scenarios for v in range(samples_per_scenario)]
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_gen_one, s, v): (s, v) for s, v in tasks}
        for fut in as_completed(futures):
            rec = fut.result()
            if rec.get('html'):
                results.append(rec)
    return results


total_target = len(PHISHING_SCENARIOS) * SAMPLES_PER_SCENARIO
print(f'Generating {total_target} synthetic phishing pages '
      f'({len(PHISHING_SCENARIOS)} scenarios × {SAMPLES_PER_SCENARIO}) '
      f'via {BEDROCK_MODEL_ID} ...')

phishing_records = generate_all_phishing(PHISHING_SCENARIOS, SAMPLES_PER_SCENARIO)
print(f'Done — {len(phishing_records)} pages generated ({total_target - len(phishing_records)} failed/skipped)')

### 2.3 · Crawl benign pages

Uses `malicious_phish.csv` as the URL seed.

In [ ]:
benign_records = crawl_benign_pages(target=BENIGN_TARGET)
print(f'Loaded {len(benign_records)} benign records')

## 3 · DOM Feature Extraction

Each HTML page is converted into a single structured text string that DistilBERT will tokenise.
Sections are prefixed with `[TITLE]`, `[META]`, `[FORMS]`, `[INPUTS]`, `[EXT_RESOURCES]`, `[LINKS]`, `[TEXT]`.

In [ ]:
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings('ignore', category=XMLParsedAsHTMLWarning)

MAX_SECTION_CHARS = 400   
MAX_BODY_CHARS    = 1000  


def extract_dom_text(html: str, url: str = '') -> str:
    """
    Extract structured text from raw HTML for transformer classification.
    Returns a single string with labelled sections.
    Sections are ordered by phishing-signal importance so the tokeniser
    truncation (at max_seq_length) cuts the least useful content last.
    """
    try:
        soup = BeautifulSoup(html, 'lxml')
    except Exception:
        soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()

    parts = []

    # [TITLE]
    title_tag = soup.find('title')
    title_txt = title_tag.get_text(strip=True)[:MAX_SECTION_CHARS] if title_tag else ''
    parts.append(f'[TITLE] {title_txt}')

    # [META] — description, keywords, author
    meta_vals = []
    for m in soup.find_all('meta', attrs={'name': True, 'content': True}):
        if m.get('name', '').lower() in ('description', 'keywords', 'author'):
            meta_vals.append(m['content'][:100])
    parts.append('[META] ' + ' | '.join(meta_vals))

    # [FORMS] — method:action pairs (phishing pages typically POST to external URLs)
    form_strs = []
    for form in soup.find_all('form'):
        action = form.get('action', '')
        method = form.get('method', 'get').upper()
        form_strs.append(f'{method}:{action[:80]}')
    parts.append('[FORMS] ' + ' | '.join(form_strs[:10]))

    # [INPUTS] — type attributes (password, hidden, email → strong phishing signal)
    input_types = [inp.get('type', 'text').lower() for inp in soup.find_all('input')]
    parts.append('[INPUTS] ' + ' '.join(input_types[:20]))

    # [EXT_RESOURCES] — scripts/iframes/images loaded from a different domain
    page_host = url.split('/')[2] if url and '//' in url else ''
    ext_srcs  = []
    for tag in soup.find_all(['script', 'iframe', 'img', 'link']):
        src = tag.get('src') or tag.get('href') or ''
        if src.startswith('http') and page_host and page_host not in src:
            ext_srcs.append(src[:80])
    parts.append('[EXT_RESOURCES] ' + ' | '.join(ext_srcs[:10]))

    # [LINKS] — href values of anchor tags
    links = [a['href'][:80] for a in soup.find_all('a', href=True)]
    parts.append('[LINKS] ' + ' | '.join(links[:20]))

    # [TEXT] — visible body text (least important for phishing signals, placed last
    # so tokeniser truncation cuts here first)
    body = soup.find('body')
    body_txt = body.get_text(separator=' ', strip=True)[:MAX_BODY_CHARS] if body else ''
    parts.append(f'[TEXT] {body_txt}')

    return ' '.join(parts)


# Quick smoke-test
if phishing_records:
    sample_out = extract_dom_text(phishing_records[90]['html'], phishing_records[90]['url'])
    print('Sample DOM text (phishing):')
    print(sample_out[:700], '...')

## 4 · Build Dataset

In [ ]:
phish_sample  = phishing_records[:PHISHING_TARGET]
benign_sample = benign_records[:BENIGN_TARGET]
all_records   = phish_sample + benign_sample
random.shuffle(all_records)

rows    = []
skipped = 0

for rec in all_records:
    try:
        text = extract_dom_text(rec['html'], rec['url'])
        if len(text.strip()) < 50:    # skip near-empty / error pages
            skipped += 1
            continue
        rows.append({'label': rec['label'], 'text': text})
    except Exception:
        skipped += 1

print(f'Total samples : {len(rows)}  (skipped {skipped})')

df = pd.DataFrame(rows)
print('Label distribution:')
print(df['label'].value_counts().to_string())

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

print(f'Train : {len(train_df)}')
print(f'Val   : {len(val_df)}')

train_df.to_csv(os.path.join(DATA_DIR, 'train.csv'),      index=False)
val_df.to_csv(  os.path.join(DATA_DIR, 'validation.csv'), index=False)
print('Saved train.csv and validation.csv')

## 5 · Upload to S3

In [ ]:
train_s3 = sess.upload_data(
    os.path.join(DATA_DIR, 'train.csv'),
    bucket=BUCKET, key_prefix=f'{PREFIX}/data'
)
val_s3 = sess.upload_data(
    os.path.join(DATA_DIR, 'validation.csv'),
    bucket=BUCKET, key_prefix=f'{PREFIX}/data'
)

print(f'train → {train_s3}')
print(f'val   → {val_s3}')

## 6 · Fine-tune with Amazon SageMaker AI HuggingFace Estimator

In [ ]:
sm_client = boto3.client('sagemaker')

JOB_PREFIX = 'html-distilbert-'

def find_existing_job(prefix):
    """Return the name + status of the most recent InProgress or Completed job matching prefix."""
    for status in ('InProgress', 'Completed'):
        resp = sm_client.list_training_jobs(
            NameContains=prefix,
            StatusEquals=status,
            SortBy='CreationTime',
            SortOrder='Descending',
            MaxResults=1,
        )
        jobs = resp.get('TrainingJobSummaries', [])
        if jobs:
            return jobs[0]['TrainingJobName'], status
    return None, None

existing_job, existing_status = find_existing_job(JOB_PREFIX)

if existing_job:
    print(f'Found existing job: {existing_job}  (status: {existing_status})')
    # To reattach to an existing job, use:
    # from sagemaker.core.resources import TrainingJob
    # job = TrainingJob.get(training_job_name=existing_job, session=sess)
    if existing_status == 'InProgress':
        print('Job is still running — reattach using TrainingJob.get() to tail logs ...')
else:

    hf_image = retrieve_image_uri(
        framework="huggingface",
        region=REGION,
        version="4.36.0",
        py_version="py310",
        base_framework_version="pytorch2.1.0",
        instance_type="ml.g5.2xlarge",
        image_scope="training",
    )

    trainer = ModelTrainer(
        sagemaker_session=sess,
        role=ROLE,
        training_image=hf_image,
        source_code=SourceCode(
            source_dir="./html_source",
            entry_script="train.py",
        ),
        compute=Compute(instance_type="ml.g5.2xlarge", instance_count=1),
        hyperparameters={
            "model_name_or_path":          "distilbert-base-uncased",
            "num_labels":                  2,
            "num_train_epochs":            3,
            "per_device_train_batch_size": 16,
            "per_device_eval_batch_size":  32,
            "learning_rate":               2e-5,
            "warmup_steps":                200,
            "weight_decay":                0.01,
            "max_seq_length":              512,
            "output_dir":                  "/opt/ml/model",
        },
        base_job_name=f"html-distilbert-{int(time.time())}",
    )

    print("Training model, will ~30mins depending on capacity")

    trainer.train(
        input_data_config=[
            InputData(channel_name="train",       data_source=train_s3),
            InputData(channel_name="validation",  data_source=val_s3),
        ],
        wait=True,
        logs="True",
    )


## 7 · Deploy to Amazon SageMaker AI Endpoint

In [ ]:
ENDPOINT_NAME = 'html-distilbert-endpoint'  # update to your desired endpoint name

deploy_image_uri = retrieve_image_uri(
    framework="huggingface",
    region=REGION,
    version="4.6.1",
    py_version="py36",
    base_framework_version="pytorch1.8.1",
    instance_type="ml.m5.xlarge",
    image_scope="inference",
)

model = Model.create(
    model_name=ENDPOINT_NAME + "-model",
    primary_container=ContainerDefinition(
        image=deploy_image_uri,
        model_data_url=trainer._latest_training_job.model_artifacts.s3_model_artifacts,
        environment={"HF_TASK": "text-classification"},
    ),
    execution_role_arn=ROLE,
    region=REGION,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=ENDPOINT_NAME + "-config",
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=ENDPOINT_NAME + "-model",
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",
            initial_variant_weight=1.0,
        )
    ],
    region=REGION,
)

endpoint = Endpoint.create(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_NAME + "-config",
    region=REGION,
)
endpoint.wait_for_status(target_status="InService")
print(f"Endpoint {ENDPOINT_NAME} is InService")

print(f'\nEndpoint live: {ENDPOINT_NAME} Copy endpoint name to use in 03_phishing_detection_pipeline.ipynb')


## 8 · Inference Helpers

In [ ]:
import boto3, json

sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)


def predict_dom(text: str) -> dict:
    payload = json.dumps({"inputs": text})
    response = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=payload,
    )
    result = json.loads(response["Body"].read())
    if isinstance(result, list):
        result = result[0]
    return result


def predict_html_page(html: str, url: str = '') -> dict:
    """End-to-end: raw HTML → prediction dict."""
    dom_text = extract_dom_text(html, url)
    return predict_dom(dom_text)


if phishing_records:
    test_rec  = phishing_records[153]
    print(test_rec)
    test_pred = predict_html_page(test_rec['html'], test_rec['url'])
    print(f'Phishing test → label={test_pred["label"]} score={test_pred["score"]:.4f}')

## 9 · Evaluate

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import pandas as pd


print(f'Running inference on {len(val_df)} validation samples ...')

has_url  = 'url' in val_df.columns
val_urls = val_df['url'].tolist() if has_url else [''] * len(val_df)

text_col = 'html' if 'html' in val_df.columns else 'text'
val_texts  = val_df[text_col].tolist()
val_labels = val_df['label'].tolist()

results = []
errors  = []

for i, (html, true_label, url) in enumerate(zip(val_texts, val_labels, val_urls)):
    try:
        extracted = extract_dom_text(html, url=url)
        pred      = predict_dom(extracted)

        raw = pred['label']
        if isinstance(raw, str):
            raw = raw.strip()
        pred_label = 1 if raw in ('LABEL_1', 'M', '1', 1) else 0
        pred_score = pred['score']

        prob_phishing = pred_score if pred_label == 1 else 1.0 - pred_score

        results.append({
            'index':              i,
            'url':                url,
            'true_label':         true_label,
            'pred_label':         pred_label,
            'pred_score':         round(pred_score, 4),
            'prob_phishing':      round(prob_phishing, 4),
            'correct':            int(true_label == pred_label),
            'error_type':         (
                'FP' if pred_label == 1 and true_label == 0 else
                'FN' if pred_label == 0 and true_label == 1 else
                'TP' if pred_label == 1 and true_label == 1 else
                'TN'
            ),
            'text_snippet':       extracted[:300],
            'extracted_features': extracted,
        })

    except Exception as e:
        errors.append({'index': i, 'url': url, 'error': str(e)})
        print(f'  [SKIP] Sample {i} failed: {e}')

print(f'Completed: {len(results)}/{len(val_df)} samples succeeded, {len(errors)} skipped.')

valid_labels = [r['true_label']    for r in results]
y_pred       = [r['pred_label']    for r in results]
y_prob       = [r['prob_phishing'] for r in results]

tn, fp, fn, tp = confusion_matrix(valid_labels, y_pred).ravel()

fpr   = fp / (fp + tn)
fnr   = fn / (fn + tp)
auroc = roc_auc_score(valid_labels, y_prob)

print(f'── Evaluation Results ──────────────────')
print(f'  Evaluated on {len(results)} / {len(val_df)} samples')
print(f'  TP     = {tp}   FP = {fp}')
print(f'  TN     = {tn}   FN = {fn}')
print(f'  FPR    = {fpr:.4f}  (false alarm rate)')
print(f'  FNR    = {fnr:.4f}  (miss rate)')
print(f'  AUROC  = {auroc:.4f}')
print()
print(classification_report(valid_labels, y_pred, target_names=['benign', 'phishing']))

results_df = pd.DataFrame(results)
results_df.to_csv('eval_predictions.csv', index=False)
print(f'Saved full predictions to eval_predictions.csv')

phishing_df = results_df[results_df['true_label'] == 1].sort_values('prob_phishing')
phishing_df.to_csv('eval_phishing_only.csv', index=False)
print(f'Saved {len(phishing_df)} phishing samples to eval_phishing_only.csv')

if errors:
    pd.DataFrame(errors).to_csv('eval_errors.csv', index=False)
    print(f'Saved {len(errors)} error records to eval_errors.csv')

fn_df = results_df[results_df['error_type'] == 'FN']
print(f'── Missed Phishing (FN) Summary ────────────────')
print(f'  Total FNs: {len(fn_df)}')
print(fn_df[['index', 'url', 'prob_phishing', 'text_snippet']].to_string(index=False))


## 10 · Cleanup

Uncomment to delete the endpoint and avoid ongoing charges.

In [ ]:
# endpoint.delete()
# endpoint_config.delete()
# model.delete()
# print(f'Endpoint {ENDPOINT_NAME} deleted.')